In [ ]:
import numpy as np
import pylab as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import psutil
from pathlib import Path
import os

import caiman as cm
from caiman.motion_correction import MotionCorrect, high_pass_filter_space
from caiman.source_extraction.cnmf import params as params
from caiman.source_extraction import cnmf
from caiman.source_extraction.cnmf.cnmf import load_CNMF

import sys
sys.path.append(os.path.abspath("code"))
from utils import download_data
import auxiliary_functions as aux

import seaborn as sns
sns.set_theme(context='notebook', style='white', font_scale=1.5)

In [ ]:
# Paths to the FOV and ROI tif files
input_tif_file_path_FOV = 'data/calcium_video.tif'
input_tif_file_path = 'data/caiman_video_trial_0.tif'

In [ ]:
# Exercise A: Select N random pixels from the ROI and plot their temporal evolution.
# Each colored box marks a randomly selected pixel region; the corresponding trace shows
# the pixel value over time. Pixels belonging to active neurons will show calcium transient dynamics.
# Run this cell multiple times to sample different pixels and appreciate the variability.

output_file = 'temporal_evolution_plot.png'
aux.temporal_evolution(file_name=input_tif_file_path, output_file_name=output_file)
plt.show()

In [ ]:
# Exercise B (histograms): The same figure produced above includes histograms of pixel values
# for each of the N randomly selected pixels (right panel). 
# Pixels from neurons show skewed distributions with a long tail toward high values 
# (due to calcium transients), while background pixels show narrow, near-Gaussian distributions 
# centered around the baseline fluorescence level.
# This difference in distribution shape can help distinguish neuron pixels from background pixels.

# Load the ROI movie once for inspection (memory-efficient: load only what is needed)
original_movie = cm.load(input_tif_file_path)
print(f'Movie shape (frames x height x width): {original_movie.shape}')
print(f'Pixel value range: [{original_movie.min():.1f}, {original_movie.max():.1f}]')

## Exercise B – Conceptual questions

**What are the differences in histograms across different regions, and how can that help distinguish a neuron pixel from a background pixel?**  
Pixels belonging to active neurons show a skewed, heavy-tailed histogram: most of the time the fluorescence is near baseline, but during calcium transients the value spikes high, producing a long right tail. Background pixels display a symmetric, narrow distribution centered on the mean fluorescence with little spread. The skewness and variance of the histogram can therefore be used as a simple statistical criterion to identify pixels with neural activity.

**If we can already see calcium traces from individual pixels, why do we need source extraction (e.g. CNMF)?**  
Working directly with single-pixel traces has several important limitations:
1. **Noise**: individual pixels have low SNR; a neuron's signal is spread over many pixels, so a single pixel captures only a fraction of the total fluorescence.
2. **Overlap**: the optical point-spread function and tissue scattering cause signals from neighbouring or overlapping neurons to mix within the same pixel. Source extraction (CNMF) explicitly models and demixes overlapping sources.
3. **Spatial context**: CNMF integrates signal over the full spatial footprint of each neuron, greatly improving SNR compared to any single pixel.
4. **Background contamination**: in 1-photon imaging especially, out-of-focus neurons contribute diffuse background fluorescence to every pixel. CNMF models this background component separately.
5. **Scalability**: a video has thousands of pixels, most of which are background. Source extraction reduces the data to a small number of meaningful neuronal traces.